# Feature Engineering

## Import

In [159]:

from pyspark.ml import Estimator, Model, Pipeline, Transformer
from pyspark.ml.feature import VectorAssembler,QuantileDiscretizer
#from synapse.ml.isolationforest import IsolationForest
from pyspark.sql.functions import first, sum, col, max, min, col, unix_timestamp, from_unixtime, lit, when, log1p, col, hour, dayofweek, to_timestamp, month
from pyspark.sql import functions as F
from pyspark.sql import SparkSession,Window
from pyspark.sql.types import IntegerType, DoubleType, StringType
import os
import seaborn as sns
import matplotlib.pyplot as plt
os.environ['HADOOP_HOME'] = "C:\\hadoop" 
# Добавляем путь к bin в системный PATH
os.environ['PATH'] += os.pathsep + "C:\\hadoop\\bin"

## ФУНКЦИЯ СБОРКИ ПАРКЕТА 

In [160]:
def finalize_parquet(folder_path, target_name):
    files = os.listdir(folder_path)
    # Добавляем [0] для получения строки
    part_file = [f for f in files if f.startswith("part-") and f.endswith(".parquet")][0]
    
    # Определяем путь к папке joined (на уровень выше временной папки)
    parent_dir = os.path.dirname(folder_path) 
    # Формируем полный путь: "../datasets/joined/train_data.parquet"
    final_path = os.path.join(parent_dir, target_name)
    
    # Если такой файл уже есть, os.rename может выдать ошибку, лучше удалить старый
    if os.path.exists(final_path):
        os.remove(final_path)

    os.rename(os.path.join(folder_path, part_file), final_path)
    
    # Очистка
    for f in os.listdir(folder_path):
        os.remove(os.path.join(folder_path, f))
    os.rmdir(folder_path)
    

In [161]:
train_files = ['../datasets/train/train_part_1.parquet','../datasets/train/train_part_2.parquet','../datasets/train/train_part_3.parquet']
train_path = '../datasets/train/'
# создание spark-сессии
spark =SparkSession.builder \
    .appName("feature_eng") \
    .config("spark.driver.memory", "5g") \
    .config("spark.executor.memory", "5g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

# загрузка исходных данных и списка релевантных для задачи признаков в объект DataFrame
df= spark.read.parquet(*train_files)

    # .config("spark.jars.packages", "com.microsoft.azure:synapseml_2.12:1.1.2") \
    # .config("spark.jars.repositories", "https://mmlspark.azureedge.net/maven") \
spark.sparkContext.setCheckpointDir("../datasets/checkpoints")

In [162]:
df.columns

['customer_id',
 'event_id',
 'event_dttm',
 'event_type_nm',
 'event_desc',
 'channel_indicator_type',
 'channel_indicator_sub_type',
 'operaton_amt',
 'currency_iso_cd',
 'mcc_code',
 'pos_cd',
 'accept_language',
 'browser_language',
 'timezone',
 'session_id',
 'operating_system_type',
 'battery',
 'device_system_version',
 'screen_size',
 'developer_tools',
 'phone_voip_call_state',
 'web_rdp_connection',
 'compromised']

## Джойн лейблов + заполнение пропусков

In [163]:
df_labels = spark.read.parquet(r'../datasets/train/train_labels.parquet')

df = df.join(other=df_labels,on=['customer_id', 'event_id'], how='left')

df = df.na.fill({
    "target": 0,
    # "label_category": "unknown"
})

# df_subset = df.join(df_labels.select('event_id'), on='event_id', how='inner')


## PREPROCESS AUF

### compromised

In [164]:
df = df.withColumn("compromised", 
    when(col("compromised").cast("byte") > 1, 1) # Привели -> Сравнили
    .otherwise(col("compromised").cast("byte"))   # Привели остальное
).fillna(-1, subset=["compromised"]) # НЕИЗВЕСТНОСТИ ЗАПОЛНИЛ  МИНУС ОДИН

### web_rdp_connection

In [165]:
df = df.fillna(-1, subset=["web_rdp_connection"]) \
       .withColumn("web_rdp_connection", col("web_rdp_connection").cast("byte"))

### phone_voip_call_state

In [166]:
df = df.fillna(-1, subset=["phone_voip_call_state"]) \
       .withColumn("phone_voip_call_state", col("phone_voip_call_state").cast("byte"))

### developer_tools

In [167]:
df = df.fillna(-1, subset=["developer_tools"]) \
       .withColumn("developer_tools", col("developer_tools").cast("byte"))

### screen_size

In [168]:
df = df.drop('screen_size') # УДАЛЕНО

### device_system_version

In [169]:
df = df.withColumn(
    'device_system_version', 
    F.regexp_extract(F.col('device_system_version'), r'(\d+)', 1).cast('int')
)
df = df.fillna(-1, subset=["device_system_version"]) \
       .withColumn("device_system_version", col("device_system_version").cast("byte"))


### battery

In [170]:
# df = df.drop('battery')

In [171]:
df = df.withColumn("battery", 
    when(col("battery") == 'not available', 0)    # Если 'not available' -> 0
    .when(col("battery").contains("%"), 1)        # Если есть процент -> 1
    .otherwise(col("battery").cast("byte"))       # В остальных случаях пытаемся в число
).fillna(-1, subset=["battery"])                  # Все пустые/невалидные -> -1

### operating_system_type

In [172]:
df = df.withColumn(
    'os_category',
    F.when(F.col('operating_system_type') == 6.0, 0)
     .when(F.col('operating_system_type') == 4.0, 1)
     .when(F.col('operating_system_type').isin([7.0, 9.0]), 2)
     .when(F.col('operating_system_type') == -1, -1) # Явно сохраняем -1
     .when(F.col('operating_system_type').isNull(), -1) # Обрабатываем null, если нужно
     .otherwise(3)
     .cast('byte')
).drop('operating_system_type')

### session_id

In [173]:
df = df.drop('session_id')

### timezone

In [174]:
df = df.withColumn(
    'tz_category',
    F.when(F.col('timezone') == -1, -1)                      # Сохраняем спецзначение
     .when(F.col('timezone') == 31, 0)                     # Условие для 0
     .when(F.col('timezone').isin([13, 21, 27, 16, 33, 20]), 1) # Группа 1
     .otherwise(2)                                           # Все остальные -> 2
     .cast('byte')                                           # Эквивалент int8
).drop('timezone')

### browser_language

In [175]:
df = df.withColumn("browser_language", 
    when(col("browser_language") == 'not available', 1) # 'not available' -> 1
    .otherwise(0)                                      # Всё остальное (любой текст или язык) -> 0
    .cast("byte")                                      # Гарантируем тип byte
)

### accept_language

In [176]:
df = df.withColumn(
    'language_category',
    F.when(F.col('accept_language').isNull(), -1)
     .when(F.lower(F.col('accept_language')) == 'ru', 0)
     .when(F.lower(F.col('accept_language')) == 'ru-ru', 1)
     # Проверяем наличие 'en' И 'ru' в строке (аналог 'en' in val and 'ru' in val)
     .when(F.lower(F.col('accept_language')).contains('en') & 
           F.lower(F.col('accept_language')).contains('ru'), 2)
     .otherwise(3)
     .cast('byte')
).drop('accept_language')

### mcc_code

In [177]:
# ПОКА ЧТО НЕТ ОБРАБОТКИ

### currency_iso_cd

In [178]:
df = df.withColumn("currency_iso_cd", 
    when(col("currency_iso_cd").cast("int") > 0, 1) # Сначала привели к числу, потом сравнили
    .otherwise(col("currency_iso_cd"))
    .cast("byte") # В самом конце гарантируем, что вся колонка — byte
).fillna(-1, subset=["currency_iso_cd"])

### operaton_amt

 СОЗДАЛ ПРИЗНАК КОТОРЫЙ ПРОПУСК В СУММЕ ОПЕРАЦИИ

In [179]:
# Если в 'other_col' пусто (NULL) -> True, иначе -> False
df = df.withColumn("operaton_amt_is_missing", 
    when(col("operaton_amt").isNull(), 1)
    .otherwise(0)
    .cast("byte")
)

ЗАПОЛНИЛ ПРОПУСКИ НУЛЯМИ

In [180]:
df = df.fillna(0, subset=["operaton_amt"])
# НЕИЗВЕСТНОСТИ ЗАПОЛНИЛ НУЛЯМИ

ПРОЛОГОРИФМИРОВАЛ СУММУ ОПЕРАЦИИ И ЗАПИСАЛ В НОВУЮ КОЛОНКУ

In [181]:
df = df.withColumn("operaton_amt_log", log1p(col("operaton_amt")))

### channel_indicator_sub_type

In [182]:
df = df.withColumn("channel_indicator_sub_type", col("channel_indicator_sub_type").cast("byte"))

### channel_indicator_type

In [183]:
df = df.withColumn("channel_indicator_type", col("channel_indicator_type").cast("byte"))

### event_dttm

In [184]:
df = df.withColumn("event_dttm", to_timestamp(col("event_dttm")))

# 2. Выделяем признаки (сразу в экономный тип byte)
df = df.withColumn("hour", hour(col("event_dttm")).cast("byte")) \
       .withColumn("day_of_week", dayofweek(col("event_dttm")).cast("byte")) \
       .withColumn("is_night", 
           when((hour(col("event_dttm")) >= 0) & (hour(col("event_dttm")) < 6), 1)
           .otherwise(0)
           .cast("byte")
       )
# df = df.drop('event_dttm')

## СРЕЗ PRETRAIN

In [185]:
df_pretrain = df.filter(
    col("event_dttm").between("2025-04-01", "2025-06-01")
)


In [186]:
end_ts  = unix_timestamp(lit(df_pretrain.select(F.max("event_dttm")).collect()[0][0]))
print(f"end_ts: {end_ts}")

end_ts: Column<'unix_timestamp(2025-05-31 23:59:59.0, 'yyyy-MM-dd HH:mm:ss')'>


In [187]:
start_ts  = unix_timestamp(lit(df_pretrain.select(F.min("event_dttm")).collect()[0][0]))
print(f"start_ts: {start_ts}")

start_ts: Column<'unix_timestamp(2025-04-01 00:00:01.0, 'yyyy-MM-dd HH:mm:ss')'>


## СРЕЗ TRAIN

In [188]:
df = df.filter(
    col("event_dttm").between("2024-10-01", "2024-11-01")
)

In [189]:
end_ts  = unix_timestamp(lit(df.select(F.max("event_dttm")).collect()[0][0]))
print(f"end_ts: {end_ts}")

end_ts: Column<'unix_timestamp(2024-11-01 00:00:00.0, 'yyyy-MM-dd HH:mm:ss')'>


In [190]:
start_ts  = unix_timestamp(lit(df.select(F.min("event_dttm")).collect()[0][0]))
print(f"start_ts: {start_ts}")

start_ts: Column<'unix_timestamp(2024-10-01 00:00:03.0, 'yyyy-MM-dd HH:mm:ss')'>


## ЗАГРУЗКА TEST PARQUET

In [191]:
test_path = "../datasets/test/test.parquet"
pretest_path = "../datasets/test/pretest.parquet"
df_test = spark.read.parquet(test_path)
df_pretest = spark.read.parquet(pretest_path)

### test preproccess

In [192]:
### compromised
df_test = df_test.withColumn("compromised", 
    when(col("compromised").cast("byte") > 1, 1) # Привели -> Сравнили
    .otherwise(col("compromised").cast("byte"))   # Привели остальное
).fillna(-1, subset=["compromised"]) # НЕИЗВЕСТНОСТИ ЗАПОЛНИЛ  МИНУС ОДИН
### web_rdp_connection
df_test = df_test.fillna(-1, subset=["web_rdp_connection"]) \
       .withColumn("web_rdp_connection", col("web_rdp_connection").cast("byte"))
### phone_voip_call_state
df_test = df_test.fillna(-1, subset=["phone_voip_call_state"]) \
       .withColumn("phone_voip_call_state", col("phone_voip_call_state").cast("byte"))
### developer_tools
df_test = df_test.fillna(-1, subset=["developer_tools"]) \
       .withColumn("developer_tools", col("developer_tools").cast("byte"))
### screen_size
df_test = df_test.drop('screen_size') # УДАЛЕНО
### device_system_version
df_test = df_test.withColumn(
    'device_system_version', 
    F.regexp_extract(F.col('device_system_version'), r'(\d+)', 1).cast('int')
)
df_test = df_test.fillna(-1, subset=["device_system_version"]) \
       .withColumn("device_system_version", col("device_system_version").cast("byte"))

### battery
# df = df.drop('battery')
df_test = df_test.withColumn("battery", 
    when(col("battery") == 'not available', 0)    # Если 'not available' -> 0
    .when(col("battery").contains("%"), 1)        # Если есть процент -> 1
    .otherwise(col("battery").cast("byte"))       # В остальных случаях пытаемся в число
).fillna(-1, subset=["battery"])                  # Все пустые/невалидные -> -1
### operating_system_type
df_test = df_test.withColumn(
    'os_category',
    F.when(F.col('operating_system_type') == 6.0, 0)
     .when(F.col('operating_system_type') == 4.0, 1)
     .when(F.col('operating_system_type').isin([7.0, 9.0]), 2)
     .when(F.col('operating_system_type') == -1, -1) # Явно сохраняем -1
     .when(F.col('operating_system_type').isNull(), -1) # Обрабатываем null, если нужно
     .otherwise(3)
     .cast('byte')
).drop('operating_system_type')
### session_id
df_test = df_test.drop('session_id')
### timezone
df_test = df_test.withColumn(
    'tz_category',
    F.when(F.col('timezone') == -1, -1)                      # Сохраняем спецзначение
     .when(F.col('timezone') == 31, 0)                     # Условие для 0
     .when(F.col('timezone').isin([13, 21, 27, 16, 33, 20]), 1) # Группа 1
     .otherwise(2)                                           # Все остальные -> 2
     .cast('byte')                                           # Эквивалент int8
).drop('timezone')
### browser_language
df_test = df_test.withColumn("browser_language", 
    when(col("browser_language") == 'not available', 1) # 'not available' -> 1
    .otherwise(0)                                      # Всё остальное (любой текст или язык) -> 0
    .cast("byte")                                      # Гарантируем тип byte
)
### accept_language
df_test = df_test.withColumn(
    'language_category',
    F.when(F.col('accept_language').isNull(), -1)
     .when(F.lower(F.col('accept_language')) == 'ru', 0)
     .when(F.lower(F.col('accept_language')) == 'ru-ru', 1)
     # Проверяем наличие 'en' И 'ru' в строке (аналог 'en' in val and 'ru' in val)
     .when(F.lower(F.col('accept_language')).contains('en') & 
           F.lower(F.col('accept_language')).contains('ru'), 2)
     .otherwise(3)
     .cast('byte')
).drop('accept_language')
### mcc_code
# ПОКА ЧТО НЕТ ОБРАБОТКИ
### currency_iso_cd
df_test = df_test.withColumn("currency_iso_cd", 
    when(col("currency_iso_cd").cast("int") > 0, 1) # Сначала привели к числу, потом сравнили
    .otherwise(col("currency_iso_cd"))
    .cast("byte") # В самом конце гарантируем, что вся колонка — byte
).fillna(-1, subset=["currency_iso_cd"])
### operaton_amt
# СОЗДАЛ ПРИЗНАК КОТОРЫЙ ПРОПУСК В СУММЕ ОПЕРАЦИИ
# Если в 'other_col' пусто (NULL) -> True, иначе -> False
df_test = df_test.withColumn("operaton_amt_is_missing", 
    when(col("operaton_amt").isNull(), 1)
    .otherwise(0)
    .cast("byte")
)
#ЗАПОЛНИЛ ПРОПУСКИ НУЛЯМИ
df_test = df_test.fillna(0, subset=["operaton_amt"])
# НЕИЗВЕСТНОСТИ ЗАПОЛНИЛ НУЛЯМИ
#ПРОЛОГОРИФМИРОВАЛ СУММУ ОПЕРАЦИИ И ЗАПИСАЛ В НОВУЮ КОЛОНКУ
df_test = df_test.withColumn("operaton_amt_log", log1p(col("operaton_amt")))
### channel_indicator_sub_type
df_test = df_test.withColumn("channel_indicator_sub_type", col("channel_indicator_sub_type").cast("byte"))
### channel_indicator_type
df_test = df_test.withColumn("channel_indicator_type", col("channel_indicator_type").cast("byte"))
### event_dttm
df_test = df_test.withColumn("event_dttm", to_timestamp(col("event_dttm")))

# 2. Выделяем признаки (сразу в экономный тип byte)
df_test = df_test.withColumn("hour", hour(col("event_dttm")).cast("byte")) \
       .withColumn("day_of_week", dayofweek(col("event_dttm")).cast("byte")) \
       .withColumn("is_night", 
           when((hour(col("event_dttm")) >= 0) & (hour(col("event_dttm")) < 6), 1)
           .otherwise(0)
           .cast("byte")
       )
# df_test = df_test.drop('event_dttm')

## PIPELINE БОГА 

In [193]:
# Класс Модели: содержит уже посчитанные данные и просто джойнит их
class PreAggAdderModel(Model):
    def __init__(self, agg_df):
        super().__init__()
        self.agg_df = agg_df

    def _transform(self, df):
        # Джойним входной датафрейм с посчитанными агрегатами
        return df.join(F.broadcast(self.agg_df), on='customer_id', how='left')

# Класс Эстиматора: вычисляет агрегаты на этапе fit
class PreAggAdder(Estimator):
    def _fit(self, df_pretrain):
        # 1. Считаем агрегаты
        agg_df = df_pretrain.groupby('customer_id').agg(
            F.mean('operaton_amt').alias('mean_operation'),
            F.percentile_approx('operaton_amt', 0.5).alias('median_operation'),
            F.count('operaton_amt').alias('user_activity')
        )
        # 2. Возвращаем модель с сохраненными агрегатами
        return PreAggAdderModel(agg_df)


class ActivityAggregator(Transformer):
    def __init__(self, days_back=7):
        super().__init__()
        self.days_back = days_back
        self.seconds_in_day = 24 * 60 * 60

    def _transform(self, df):
        # 1. Приводим время к числу
        time_col = F.col("event_dttm").cast("long")
        
        # 2. Окно (используем только нужные колонки)
        window_spec = Window.partitionBy("customer_id") \
                            .orderBy(time_col) \
                            .rangeBetween(-self.days_back * self.seconds_in_day, -1)

        # 3. Добавляем колонки (используем percentile_approx вместо median)
        df = df.withColumn("avg_ops_last_week", F.avg("operaton_amt").over(window_spec)) \
               .withColumn("count_ops_last_week", F.count("operaton_amt").over(window_spec))\
                .withColumn("quantiles", F.percentile_approx("operaton_amt", [0.25, 0.5, 0.75]).over(window_spec))

        # 2. Распаковываем массив в отдельные признаки
        df = df.withColumn("median_ops_last_week", F.col("quantiles")[1]) \
            .withColumn("inter_quantile_range", F.col("quantiles")[2] - F.col("quantiles")[0])

        # 3. Удаляем временную колонку с массивом
        df = df.drop("quantiles")
        
        return df.fillna(0, subset=["avg_ops_last_week", "count_ops_last_week", "median_ops_last_week",'inter_quantile_range'])
    
class TimeWindowAggregator(Transformer):
    def __init__(self, hours_back=2):
        super().__init__()
        self.hours_back = hours_back
        self.seconds_in_hour = 3600

    def _transform(self, df):
        time_col = F.col("event_dttm").cast("long")
        
        # Окно: от -2 часов до текущего момента (не включая текущую транзакцию -1 сек)
        # Если хочешь ВКЛЮЧАТЬ текущую — ставь 0 в конце.
        window_2h = Window.partitionBy("customer_id") \
                          .orderBy(time_col) \
                          .rangeBetween(-self.hours_back * self.seconds_in_hour, -1)

        # 1. Считаем количество операций (самый важный признак для фрода "набегами")
        df = df.withColumn("ops_count_2h", F.count("operaton_amt").over(window_2h))
        
        # 2. Сумма операций за 2 часа (помогает ловить быстрый вывод средств)
        df = df.withColumn("ops_sum_2h", F.sum("operaton_amt").over(window_2h))

        # 3. Максимальный чек за эти 2 часа
        df = df.withColumn("max_amt_2h", F.max("operaton_amt").over(window_2h))

        # Очищаем null, если за 2 часа операций не было
        return df.fillna(0, subset=["ops_count_2h", "ops_sum_2h", "max_amt_2h"])

## ВКЛ И ВЫКЛ ПАЙПЛАЙНА

In [194]:


# --- ШАГ 1: ТРЕНИРОВКА (Валидация) ---
# Создаем агрегатор, который видит только претрейн
agg_train = PreAggAdder()
act_train = ActivityAggregator(days_back=2)
time_train = TimeWindowAggregator()
pipeline_train = Pipeline(stages=[agg_train,act_train,time_train])

# Обучаем и применяем (Transform)
model_train = pipeline_train.fit(df_pretrain)
df = model_train.transform(df)

# --- ШАГ 2: ТЕСТ (Продакшен-режим) ---
# Создаем ТАКОЙ ЖЕ агрегатор, но с ОБНОВЛЕННОЙ историей
agg_test = PreAggAdder()
act_test = ActivityAggregator(days_back=2)
time_test = TimeWindowAggregator()
pipeline_test = Pipeline(stages=[agg_test,act_test,time_test])

# Обучаем заново (fit), чтобы зафиксировать новую таблицу агрегатов
model_test = pipeline_test.fit(df_pretest)
df_test = model_test.transform(df_test)

## ДЕЛАЕМ СОХРАНЕНИЕ

In [195]:
df = df.localCheckpoint()
df_test = df_test.localCheckpoint()
print(f"Строк в трейне после трансформации: {df.count()}") 
print(f"Строк в трейне после трансформации: {df_test.count()}") 

Строк в трейне после трансформации: 8972204
Строк в трейне после трансформации: 633683


## ПОСЛЕ ОТРАБОТКИ ПАЙПЛАЙНА ДРОПАЕМ ВРЕМЯ

In [196]:
df = df.drop('event_dttm')
df_test = df_test.drop('event_dttm')

## ЗАГЛУШКА КОТОРАЯ ВСЕ ПРОПУСКИ УБИРАЕТ!!!!!!!!

In [197]:
df = df.fillna(-1)
df_test = df_test.fillna(-1)

## ВЕЛИКОЕ ОЧИЩЕНИЕ 

In [198]:
# # 2. Настройка: указываем логарифмированную колонку и категории
# num_col = "operaton_amt_log" 
# cat_cols =  ["event_type_nm",
#               "event_desc",
#                 "channel_indicator_type",
#                 'channel_indicator_sub_type',
# ]

# #СЮДА ПЕРЕДАВАТЬ ВАЖНОЕ, МАСТХЭВ, ТО ЧТО ОПРЕДЕЛЯЕТ СУЩНОСТЬ
# output_bucket_col = "bucket_log"

In [199]:
# # 3. Настройка дискретизатора
# discretizer = QuantileDiscretizer(
#     numBuckets=1000, 
#     inputCol=num_col, 
#     outputCol=output_bucket_col,
#     relativeError=0.01 
# )

In [200]:
# # 4. Обучение и трансформация
# # Spark просканирует данные, найдет границы квантилей и присвоит каждой строке ID бакета
# df_prepared = discretizer.fit(df).transform(df)


In [201]:
# # 5. Дедупликация
# # Оставляем одну уникальную запись на комбинацию (бакет цены + ваши категории)
# df_final = df_prepared.dropDuplicates([output_bucket_col] + cat_cols)

In [202]:
# # 6. Очистка временных колонок
# # Убираем bucket_log, чтобы оставить исходные 20 колонок
# df_final = df_final.drop(output_bucket_col)

In [203]:
# target_all = df.select('target').filter('target==1').count()
# target_final = df_final.select('target').filter('target==1').count()
# all_rows = df.count()
# final_rows = df_final.count()

In [204]:
# print('СТАТИСТИЧЕСКАЯ ВЫВОДКА')
# print(f'Общее кол-во строк ALL: {all_rows}, фродов в ALL: {target_all}, частота появления фрода ALL {(target_all/all_rows)*100:.5f}%')
# print(f'Общее кол-во строк FINAL: {final_rows}, фродов в FINAL: {target_final}, частота появления фрода FINAL {(target_final/final_rows)*100:.5f}%')
# print(f'РАЗМЕРНОСТЬ СОКРАТИЛАСЬ ПО СТРОКАМ в {all_rows/final_rows:.3f} раз, ПО ФРОДУ В {target_all/target_final:.3f} раз')


## ВОЗРАЩЕНИЕ ФРОДО

In [205]:

# # 4. ОБЯЗАТЕЛЬНО: обрываем связи перед записью
# df_final = df_final.localCheckpoint()
# print(f"Итого после склейки: {df_final.count()}")

## СРАВНЕНИЕ ИСХОДНОГО ПРЕПРОЦЕССНУТОГО И СВЕТЛОГО ФИЛЬТРОВАННОГО 

In [206]:
def plot_compare_distributions(df1, df2, columns, label1='FULL', label2='FILTER'):
    # 1. Берем выборку (sampling), чтобы не вешать драйвер, если данных миллионы
    # Если данных мало, можно просто .toPandas()
    p1 = df1.select(columns).sample(0.1).toPandas()
    p2 = df2.select(columns).toPandas()
    
    # 2. Настраиваем сетку графиков
    num_cols = len(columns)
    fig, axes = plt.subplots(nrows=(num_cols + 1) // 2, ncols=2, figsize=(15, num_cols * 2))
    axes = axes.flatten()

    for i, col in enumerate(columns):
        # Рисуем плотность для обоих наборов на одном сабплоте
        sns.histplot(p1[col], ax=axes[i], label=label1, fill=True, alpha=0.3,kde=True,stat='density')
        sns.histplot(p2[col], ax=axes[i], label=label2, fill=True, alpha=0.3,kde=True,stat='density')
        
        axes[i].set_title(f'Distribution of {col}')
        axes[i].legend()

    plt.tight_layout()
    plt.show()

In [207]:
# Использование:
cols_to_check = [       
        "web_rdp_connection",
        "is_night",
        "os_category",
        "operaton_amt_log",
        "phone_voip_call_state",
        # "customer_id",
        'month',
        "day_of_week",
        "event_desc",
        "developer_tools",
        "hour",
        "event_type_nm",
        "compromised",
        "channel_indicator_sub_type",
        "channel_indicator_type",
        # "tz_category",
        "pos_cd"
]
# plot_compare_distributions(df, df_final, cols_to_check)

## ОБЫЧНО БАЗОВЫЙ ДРОП КОЛОНОК НУ..

In [208]:
df = df.drop('customer_id','operaton_amt')

## РАЗДЕЛЕНИЕ НА ТРЕНИРОВОЧНУЮ И ВАЛИДАЦИОННУЮ 

In [209]:
fractions = {row[0]: 0.8 for row in df.select("target").distinct().collect()}

df_train = df.stat.sampleBy("target", fractions, seed=42)
df_valid = df.join(df_train, on="event_id", how="left_anti")


## ВЫГРУЗКА ФАЙЛОВ АЙОУ

In [210]:
# Пути для промежуточных (многофайловых) данных
multi_temp = "../datasets/joined/multi_temp"

# Список пар (датафрейм, имя финального файла, временная папка для 1-го файла)
datasets_to_save = [
    (df_train, "train_data.parquet", "../datasets/joined/temp_train_folder"),
    (df_valid, "valid_data.parquet", "../datasets/joined/temp_valid_folder"),
    (df_test, "test_data.parquet", "../datasets/joined/temp_test_folder")
]

for df, final_name, single_folder in datasets_to_save:
    print(f"Обработка {final_name}...")
    
    # ШАГ 1: Записываем результат тяжелых трансформаций (Window, Join) 
    # в многопоточном режиме (без repartition(1)). Это не уронит Spark.
    df.write.mode("overwrite").parquet(multi_temp)
    
    spark.catalog.clearCache() 
    # ШАГ 2: Читаем уже готовые вычисленные данные и сжимаем в 1 файл.
    # Теперь это просто перекладывание байтов, ресурсов нужно в разы меньше.
    spark.read.parquet(multi_temp) \
         .repartition(1) \
         .write.mode("overwrite") \
         .parquet(single_folder)
    
    # ШАГ 3: Ваша функция переименования (теперь она сработает)
    finalize_parquet(single_folder, final_name)
    
    print(f"Файл {final_name} успешно создан.")

# Очистка общей временной папки
import shutil
if os.path.exists(multi_temp):
    shutil.rmtree(multi_temp)

print("\nВсе данные успешно сохранены!")

Обработка train_data.parquet...
Файл train_data.parquet успешно создан.
Обработка valid_data.parquet...
Файл valid_data.parquet успешно создан.
Обработка test_data.parquet...
Файл test_data.parquet успешно создан.

Все данные успешно сохранены!


## ВЫХОДНОЙ СПИСОК КОЛОНОК 

In [211]:
df.columns

['customer_id',
 'event_id',
 'event_type_nm',
 'event_desc',
 'channel_indicator_type',
 'channel_indicator_sub_type',
 'operaton_amt',
 'currency_iso_cd',
 'mcc_code',
 'pos_cd',
 'browser_language',
 'battery',
 'device_system_version',
 'developer_tools',
 'phone_voip_call_state',
 'web_rdp_connection',
 'compromised',
 'os_category',
 'tz_category',
 'language_category',
 'operaton_amt_is_missing',
 'operaton_amt_log',
 'hour',
 'day_of_week',
 'is_night',
 'mean_operation',
 'median_operation',
 'user_activity',
 'avg_ops_last_week',
 'count_ops_last_week',
 'median_ops_last_week',
 'inter_quantile_range',
 'ops_count_2h',
 'ops_sum_2h',
 'max_amt_2h']

In [212]:
set_cols = set(df.columns)
set_cols = set_cols - set(['target','event_id'])
cat_cols = set_cols - set(['operaton_amt',
                            'operaton_amt_log',
                            'user_activity',
                            'median_operation',
                            'user_activity',
                            'avg_ops_last_week',
                            'count_ops_last_week',
                            'median_ops_last_week',
                            'inter_quantile_range',
                            "ops_count_2h", 
                            "ops_sum_2h", 
                            "max_amt_2h",
                            'mean_opeartion'])

output_cols = {}
output_cols['all'] = list(set_cols)
output_cols['cat'] = list(cat_cols)
output_cols['target'] = 'target'
output_cols['id'] = 'event_id'

In [213]:
import json
with open('../datasets/joined/columns_list.json', "w", encoding="utf-8") as f:
    json.dump(output_cols, f, ensure_ascii=False, indent=4)